# Encodage des 375 580 essais avec EmbeddingGemma 2 (expérience 1, étape 5b)

Texte encodé : version « complet » (titre + conditions + résumé + description + critères), le même que l'index BM25. Calcul en float32 (le plus rapide et exact sur T4), vecteurs normalisés, stockés en float16 sur Google Drive par morceaux de 10 000 essais.

**À chaque session** : *Exécution > Modifier le type d'exécution > GPU T4*, puis *Exécution > Tout exécuter*, et autoriser l'accès à Google Drive. Si la session se coupe, relancer de la même façon : le carnet reprend au premier morceau manquant. Il s'arrête de lui-même quand les 38 morceaux sont faits.

Sortie : `Mon Drive/hybrid_ir/exp1_embeddings/` (≈ 600 Mo).

In [ ]:
# 1. GPU et Google Drive
import os, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # moins de mémoire GPU perdue en fragments
import torch
assert torch.cuda.is_available(), "Pas de GPU : choisir GPU T4 dans Exécution > Modifier le type d'exécution"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
OUT = Path("/content/drive/MyDrive/hybrid_ir/exp1_embeddings"); OUT.mkdir(parents=True, exist_ok=True)
print("sauvegardes :", OUT, "|", len(list(OUT.glob("chunk_*.npy"))), "morceaux déjà faits")

In [ ]:
# 2. Bibliothèques aux versions du test du 03/11/2026
!pip install -q "sentence-transformers==6.2.0" "transformers==5.22.1" "ir_datasets==0.6.3"
import sentence_transformers, transformers, ir_datasets
print(sentence_transformers.__version__, transformers.__version__, ir_datasets.__version__)

In [ ]:
# 3. Corpus TREC CT 2021 (téléchargé par Colab, ≈ 1,8 Go) et conditions lues dans le XML brut
import os, zipfile, xml.etree.ElementTree as ET
os.environ["IR_DATASETS_HOME"] = "/content/irds"
dataset = ir_datasets.load("clinicaltrials/2021/trec-ct-2021")
docs = list(dataset.docs_iter())
XML_DIR = Path("/content/xml"); XML_DIR.mkdir(exist_ok=True)
archives = []
for part in range(1, 6):  # ir_datasets ne garde pas les archives XML : on les télécharge depuis TREC
    path = XML_DIR / f"ClinicalTrials.2021-04-27.part{part}.zip"
    if not path.exists():
        url = f"http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part{part}.zip"
        subprocess.run(["curl", "-sSfL", "-o", str(path), url], check=True)  # le serveur refuse urllib (406)
    archives.append(path)
print(len(archives), "archives :", [a.name for a in archives])
conditions = {}
for archive in archives:
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            if name.endswith(".xml"):
                root = ET.fromstring(z.read(name))
                nct_id = (root.findtext("id_info/nct_id") or "").strip()
                conditions[nct_id] = [t for n in root.findall("condition") if (t := (n.text or "").strip())]
n_with = sum(bool(c) for c in conditions.values())
print(len(docs), "essais ;", n_with, "avec conditions")
assert len(docs) == 375580 and n_with == 374720, "effectifs différents de ceux mesurés sur le PC : s'arrêter et prévenir"


In [ ]:
# 4. Texte « complet », format document de la fiche du modèle (identique au test du 03/11)
def compose(doc):
    parts = ["; ".join(conditions.get(doc.doc_id, [])), doc.summary or "", doc.detailed_description or "", doc.eligibility or ""]
    body = " ".join(" ".join(p.split()) for p in parts if p)
    return f"title: {doc.title or 'none'} | text: {body}"
ids = [d.doc_id for d in docs]
texts = [compose(d) for d in docs]
del docs
import json
(OUT / "doc_ids.json").write_text(json.dumps(ids))
print(texts[0][:300])

In [ ]:
# 5. Modèle : révision figée du test, texte seul, float32
from sentence_transformers import SentenceTransformer
MODEL, REVISION = "google/embeddinggemma-2", "f9972ac22e696a105c482202646979e33e4b5a1e"
model = SentenceTransformer(MODEL, revision=REVISION, device="cuda", model_kwargs={"torch_dtype": torch.float32},
                            config_kwargs={"vision_config": None, "audio_config": None})
(OUT / "manifest.json").write_text(json.dumps({"model": MODEL, "revision": REVISION, "dtype_compute": "float32", "dtype_stored": "float16",
    "text": "title: {titre} | text: {conditions ; } {résumé} {description} {critères}", "n_trials": len(ids), "chunk_size": 10000,
    "versions": {"torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__, "transformers": transformers.__version__, "ir_datasets": ir_datasets.__version__}}, indent=1))

In [ ]:
# 6. Encodage par morceaux de 10 000, avec reprise ; chaque morceau est écrit sur Drive dès qu'il est fini
import time, numpy as np
CHUNK = 10000
n_chunks = (len(texts) + CHUNK - 1) // CHUNK

def encode(docs, max_batch=16):
    """Textes triés du plus long au plus court ; seul un lot qui ne tient pas en mémoire est réduit,
    et la taille remonte ensuite vers 16 pour les textes plus courts."""
    order = sorted(range(len(docs)), key=lambda k: len(docs[k]), reverse=True)
    out = np.zeros((len(docs), model.get_sentence_embedding_dimension()), dtype=np.float32)
    position, batch_size = 0, max_batch
    while position < len(order):
        batch = order[position:position + batch_size]
        try:
            out[batch] = model.encode([docs[k] for k in batch], batch_size=len(batch), normalize_embeddings=True, convert_to_numpy=True)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size //= 2
            continue
        position += len(batch)
        batch_size = min(max_batch, batch_size * 2)
    return out

for i in range(n_chunks):
    path = OUT / f"chunk_{i:03d}.npy"
    if path.exists():
        continue
    start = time.perf_counter()
    emb = encode(texts[i * CHUNK:(i + 1) * CHUNK])
    assert not np.isnan(emb).any(), f"NaN dans le morceau {i}"
    np.save(OUT / f"tmp_{i:03d}.npy", emb.astype(np.float16))
    (OUT / f"tmp_{i:03d}.npy").rename(path)  # écriture puis renommage : pas de morceau à moitié écrit
    print(f"morceau {i + 1}/{n_chunks} : {len(emb)} essais en {time.perf_counter() - start:.0f} s", flush=True)
done = len(list(OUT.glob("chunk_*.npy")))
print("TERMINÉ" if done == n_chunks else f"{done}/{n_chunks} morceaux")